In [1]:
"""Smoke-test for the NEW Rossmann forecasting pipeline (hypertuning).

Runs the full feature-engineering -> target-transform -> inner-CV objective and
refit flow on a few real stores, then exercises hypertuning.optimize end-to-end
to validate the updated TimeSeriesCV(n_splits, train_size, test_size) API.
"""
import os
import logging
import optuna
import pandas as pd

from src import features
from src.preprocessing import preprocess_data
from src.engine.target_transformer import TargetTransformer
from src.settings import AppSettings

config = AppSettings.from_yaml('./config.yaml')

os.makedirs(config.path.log_dir, exist_ok=True)
assert os.path.exists(config.path.data_dir), \
    f"Data directory {config.path.data_dir} does not exist."

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.basicConfig(level=logging.INFO,
                    filename=config.path.logs,
                    format="%(levelname)s  %(message)s")

logger = logging.getLogger(__name__)




c:\Users\m_kal\Downloads\rossmann_store_sales\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sales = pd.read_csv(config.path.train)
stores = pd.read_csv(config.path.stores)

stores_to_use = [1, 2, 3]
stores = stores[stores['Store'].isin(stores_to_use)]

df = preprocess_data(sales, stores)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_15564\288647734.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv(config.path.train)


In [3]:
y = df.set_index(['Store', 'Date'])['Sales']

X = (df
     .set_index(['Date'])
     .groupby('Store')
     .apply(lambda df: features.compute(df, config.feature_engineering, config.horizon)))

trf = TargetTransformer(forecast_horizon=pd.DateOffset(days=-config.horizon.days),
                        anchor_col='lag_days_0')
trf.fit(X)

y = trf.transform(y)    # forward difference target
X = X.loc[y.index]      # align features with target

C:\Users\m_kal\AppData\Local\Temp\ipykernel_15564\1219067114.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: features.compute(df, config.feature_engineering, config.horizon)))


In [4]:
X.columns

Index(['lag_days_0', 'lag_days_1', 'lag_days_2', 'lag_days_3', 'lag_days_4',
       'lag_days_5', 'lag_days_6', 'lag_days_7', 'lag_days_8', 'lag_days_9',
       'lag_days_10', 'lag_days_11', 'lag_days_12', 'lag_days_13',
       'lag_days_14', 'lag_weeks_3', 'lag_weeks_4', 'lag_months_2',
       'lag_months_3', 'lag_months_4', 'lag_months_5', 'lag_months_6',
       'lag_years_1', 'diff_days_1', 'diff_days_2', 'diff_days_3',
       'diff_days_4', 'diff_days_5', 'diff_days_6', 'diff_days_14',
       'diff_days_30', 'rolling_mean_7D', 'rolling_mean_14D',
       'rolling_mean_30D', 'rolling_mean_60D', 'year', 'quarter', 'month',
       'week_of_month', 'day_of_week', 'is_month_start', 'is_weekend',
       'is_weekday', 'is_month_end', 'competition_days_since_start',
       'pre_holiday_wave', 'post_holiday_wave', 'Promo', 'Promo2'],
      dtype='object')

In [5]:
X[["pre_holiday_wave", "post_holiday_wave"]]

pre_holiday_wave  post_holiday_wave
Store Date                                           
1     2013-01-01               0.0       4.393693e-02
      2013-01-02               0.0       2.279418e-02
      2013-01-03               0.0       1.110900e-02
      2013-01-04               0.0       5.086069e-03
      2013-01-05               0.0       2.187491e-03
...                            ...                ...
3     2015-07-17               NaN       7.535073e-39
      2015-07-18               NaN       2.660211e-40
      2015-07-19               NaN       8.822575e-42
      2015-07-20               NaN       2.746545e-43
      2015-07-21               NaN       8.407791e-45

[2796 rows x 2 columns]